# FLOW — Vocabulary Growth (Corpus-Grounded)

**Geometric Causal Architecture** — weight-free, token-free reasoning.  
Knowledge stored as shape in a 104D Riemannian manifold.

This notebook **loads the K-12 model** as a base and feeds **5 real-world corpora**  
(2,000 texts each = 10,000 total) through the VocabularyBuilder PMI pipeline  
to grow a **semantically grounded** vocabulary.

### Why this matters

The K-12 model places words by character n-grams (spelling similarity).  
This gives NO semantic structure — "cat" and "car" end up near each other  
because they share letters, not meaning.

**PMI co-occurrence from real text** fixes this: words that appear together  
in actual sentences get pulled together on the manifold. "cat" moves near  
"kitten" and "pet", while "car" moves near "drive" and "road".

### Corpus mix

| Dataset | Texts | Domain |
|---|---|---|
| Simple English Wikipedia | 2,000 | General knowledge |
| OpenAssistant (oasst1) | 2,000 | Agent dialogue |
| Alpaca (cleaned) | 2,000 | Instruction following |
| Glaive Function Calling v2 | 2,000 | Tool calling |
| SlimOrca | 2,000 | Reasoning chains |

**Hardware:** Kaggle CPU (4 cores, 30 GB RAM)  
**Estimated runtime:** 10–20 minutes  
**Base model:** K-12 pipeline (`FLOW-K12_manifold.npz` + `FLOW-K12_vocab.npz`)  
**Output:** `FLOW-Grounded_manifold.npz` + `FLOW-Grounded_vocab.npz`

## 1. Setup — Clone/refresh repo & install dependencies

In [ ]:
import os, subprocess

# ── GitHub PAT from Kaggle Secrets ──────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    PAT = UserSecretsClient().get_secret("GITHUB_PAT")
    print(f"✓ PAT loaded from Kaggle Secrets ({len(PAT)} chars)")
except Exception:
    PAT = None
    print("⚠ No Kaggle Secrets — using public clone (push disabled)")

REPO_URL = f"https://{PAT}@github.com/Unseengap/FLOW.git" if PAT else "https://github.com/Unseengap/FLOW.git"
REPO_DIR = "FLOW"

# ── Clone or refresh ──────────────────────────────────────────
if os.path.isdir(REPO_DIR):
    print("Existing FLOW clone detected — refreshing...")
    before = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    # Update remote URL (in case PAT changed)
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "fetch", "origin", "main"],
                   cwd=REPO_DIR, capture_output=True, text=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"],
                   cwd=REPO_DIR, capture_output=True, text=True)
    after = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    if before == after:
        print(f"  Already up-to-date ({after})")
    else:
        log = subprocess.run(
            ["git", "--no-pager", "log", "--oneline", f"{before}..{after}"],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        print(f"  Updated {before} → {after}")
        for line in log.stdout.strip().split("\n")[:10]:
            if line.strip():
                print(f"    {line}")
else:
    print("Cloning FLOW repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                   capture_output=True, text=True, check=True)
    after = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    print(f"  ✓ Cloned at {after}")

assert os.path.isdir("FLOW/src"), "FLOW repo not found"
print(f"✓ Repository ready")

In [ ]:
# Install dependencies (pure math — no ML frameworks)
!pip install numpy scipy networkx datasets -q

# Optional acceleration
!pip install faiss-cpu -q 2>/dev/null || echo "FAISS not available — linear scan fallback"

print("✓ Dependencies installed")

In [ ]:
# Add FLOW to Python path
import sys
sys.path.insert(0, "FLOW")

# Core imports
from src.phase5 import GEOPipeline, PipelineResult
from src.vocabulary import VocabularyBuilder, VocabularyStore
from src.vocabulary.cooccurrence import CoOccurrenceCounter, CoOccurrenceMatrix
from src.vocabulary.contrast_scheduler import ContrastScheduler, ContrastPair
from src.phase2.contrast_engine.engine import JudgmentType
from src.persistence import ManifoldSnapshot
from src.phase3.annealing_engine.experience import Experience
from src.phase1.expression.matcher import ExpressionEntry
import numpy as np
import time

print("✓ All FLOW imports successful")
print(f"  numpy {np.__version__}")

## 2. Load K-12 Base Model

Start from the K-12 pipeline model which has curriculum concepts  
from Kindergarten through Grade 12. Falls back to the V1 base model  
if K-12 is not available.

In [ ]:
# ── Locate best available base model ───────────────────────
MODELS_DIR = f"{REPO_DIR}/models"

# Prefer K-12 model, fall back to V1 base
if os.path.isfile(f"{MODELS_DIR}/FLOW-K12_manifold.npz"):
    BASE_MANIFOLD = f"{MODELS_DIR}/FLOW-K12_manifold.npz"
    BASE_VOCAB    = f"{MODELS_DIR}/FLOW-K12_vocab.npz"
    base_name = "K-12 pipeline"
elif os.path.isfile(f"{MODELS_DIR}/FLOW-V1-Base_manifold.npz"):
    BASE_MANIFOLD = f"{MODELS_DIR}/FLOW-V1-Base_manifold.npz"
    BASE_VOCAB    = f"{MODELS_DIR}/FLOW-V1-Base_vocab.npz"
    base_name = "V1 base"
else:
    raise FileNotFoundError(
        f"No model found in {MODELS_DIR}/. "
        "Run kaggle_base_model.ipynb or kaggle_k12_pipeline.ipynb first."
    )

print(f"Loading {base_name} model...")
print(f"  Manifold : {BASE_MANIFOLD}")
print(f"  Vocab    : {BASE_VOCAB}")

# ── Load pipeline ─────────────────────────────────────────
# T0=1.0 : high initial temperature for flexible placement
# lambda_=0.005 : slower cooling for large-scale ingestion
# T_floor=0.03  : lower floor to allow crystallisation over time
pipeline = GEOPipeline.load(
    BASE_MANIFOLD,
    vocabulary_path=BASE_VOCAB,
    T0=1.0,
    lambda_=0.005,
    T_floor=0.03,
    flow_seed=42,
)

# Record base vocabulary for later merging
base_vocab_entries = VocabularyStore.load(BASE_VOCAB)
n_base_vocab = len(base_vocab_entries)

print(f"\n✓ {base_name} model loaded")
print(f"  Manifold dimension : {pipeline.manifold.DIM}")
print(f"  Base concepts      : {pipeline.n_concepts:,}")
print(f"  Base vocab entries : {n_base_vocab:,}")
print(f"  Temperature        : {pipeline.temperature:.4f}")

## 3. Load Corpora — 5 datasets × 2,000 texts

Five complementary datasets that ground FLOW in real language:
- **Wikipedia** — broad concept coverage, clean prose
- **OpenAssistant** — conversational Q&A patterns
- **Alpaca** — imperative/procedural instructions
- **Glaive Functions** — structured tool-calling language
- **SlimOrca** — step-by-step reasoning chains

In [ ]:
from datasets import load_dataset
from collections import Counter

all_texts = []   # collect (text, source_tag) pairs

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
N_PER_DATASET = 2_000     # 5 datasets × 2K = 10K total texts
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 3A. Wikipedia — Simple English (general knowledge)
print("Loading Simple English Wikipedia...")
ds_wiki = load_dataset("wikimedia/wikipedia", "20231101.simple",
                       split="train", trust_remote_code=True)
for i, article in enumerate(ds_wiki):
    if i >= N_PER_DATASET:
        break
    text = article.get("text", "")
    if text and len(text) > 50:
        all_texts.append((text, "wikipedia"))
print(f"  ✓ Wikipedia: {len([t for t in all_texts if t[1]=='wikipedia']):,}")

# 3B. OpenAssistant (oasst1) — dialogue
print("Loading OpenAssistant (oasst1)...")
ds_oasst = load_dataset("OpenAssistant/oasst1", split="train",
                        trust_remote_code=True)
for i, row in enumerate(ds_oasst):
    if i >= N_PER_DATASET:
        break
    text = row.get("text", "")
    if text and len(text) > 30:
        all_texts.append((text, "openassistant"))
print(f"  ✓ OpenAssistant: {len([t for t in all_texts if t[1]=='openassistant']):,}")

# 3C. Alpaca (cleaned) — instruction following
print("Loading Alpaca (cleaned)...")
ds_alpaca = load_dataset("yahma/alpaca-cleaned", split="train",
                         trust_remote_code=True)
for i, row in enumerate(ds_alpaca):
    if i >= N_PER_DATASET:
        break
    parts = []
    if row.get("instruction"): parts.append(row["instruction"])
    if row.get("input"): parts.append(row["input"])
    if row.get("output"): parts.append(row["output"])
    text = " ".join(parts)
    if text and len(text) > 30:
        all_texts.append((text, "alpaca"))
print(f"  ✓ Alpaca: {len([t for t in all_texts if t[1]=='alpaca']):,}")

# 3D. Glaive Function Calling v2 — tool calling
print("Loading Glaive Function Calling v2...")
ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2",
                         split="train", trust_remote_code=True)
for i, row in enumerate(ds_glaive):
    if i >= N_PER_DATASET:
        break
    text_parts = []
    if row.get("system"): text_parts.append(row["system"])
    if row.get("chat"): text_parts.append(row["chat"])
    text = " ".join(text_parts)
    if text and len(text) > 30:
        all_texts.append((text, "glaive_functions"))
print(f"  ✓ Glaive Functions: {len([t for t in all_texts if t[1]=='glaive_functions']):,}")

# 3E. SlimOrca — reasoning chains
print("Loading SlimOrca...")
ds_orca = load_dataset("Open-Orca/SlimOrca", split="train",
                       trust_remote_code=True)
for i, row in enumerate(ds_orca):
    if i >= N_PER_DATASET:
        break
    convos = row.get("conversations", [])
    text = " ".join(
        turn.get("value", "") for turn in convos if isinstance(turn, dict)
    )
    if text and len(text) > 30:
        all_texts.append((text, "slimorca"))
print(f"  ✓ SlimOrca: {len([t for t in all_texts if t[1]=='slimorca']):,}")

# Summary
source_counts = Counter(t[1] for t in all_texts)
print(f"\n{'='*60}")
print(f"Total texts loaded: {len(all_texts):,}")
for src, cnt in source_counts.most_common():
    print(f"  {src:20s} : {cnt:>6,}")
print(f"{'='*60}")

## 4. Configure VocabularyBuilder, Checkpoint System & Feed Corpora

In [ ]:
# ── Configuration ─────────────────────────────────────────────
V_MAX             = 10_000   # vocabulary ceiling
MIN_COUNT         = 3        # lower threshold for 10K corpus
WINDOW_SIZE       = 5        # co-occurrence window
N_CONTRAST_PASSES = 2        # C4 refinement passes
TAU_SAME          = 1.0      # PMI threshold for SAME
TAU_DIFF          = -0.5     # PMI threshold for DIFFERENT
BATCH_SIZE        = 512      # contrast batch size

print(f"Configuration:")
print(f"  V_MAX             = {V_MAX:,}")
print(f"  MIN_COUNT         = {MIN_COUNT}")
print(f"  WINDOW_SIZE       = {WINDOW_SIZE}")
print(f"  N_CONTRAST_PASSES = {N_CONTRAST_PASSES}")
print(f"  Total texts       = {len(all_texts):,}")

# ══════════════════════════════════════════════════════════════════════════════
# Checkpoint / Resume Infrastructure
# ══════════════════════════════════════════════════════════════════════════════
#
# PROBLEM: Kaggle destroys /kaggle/working/ when the session ends.
# SOLUTION: Every checkpoint artifact is stored in models/ (tracked by git)
# and pushed to GitHub after each step.  On a fresh Kaggle session the repo
# clone already contains the artifacts, so we copy them back to /kaggle/working/
# and resume from where we left off.
#
# Artifacts pushed to GitHub (survive across sessions):
#   models/FLOW-Grounded_manifold.npz  — full manifold state (M(t))
#   models/FLOW-Grounded_vocab.npz     — final vocabulary entries
#   models/FLOW-Grounded_pmi.npz       — PMI matrix (compact numpy format)
#   models/FLOW-Grounded_entries.npz   — pre-merge expression entries
#   checkpoints/*.json                 — step-completion markers + fingerprint
#
# Nothing important stays only in /kaggle/working/.
# ══════════════════════════════════════════════════════════════════════════════

import hashlib, json, shutil

OUTPUT_DIR = "/kaggle/working"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── File paths (all artifacts that must survive across sessions) ──────────
GROUNDED_MANIFOLD_PATH = f"{OUTPUT_DIR}/FLOW-Grounded_manifold.npz"
GROUNDED_VOCAB_PATH    = f"{OUTPUT_DIR}/FLOW-Grounded_vocab.npz"
PMI_CACHE_PATH         = f"{OUTPUT_DIR}/FLOW-Grounded_pmi.npz"
ENTRIES_CACHE_PATH     = f"{OUTPUT_DIR}/FLOW-Grounded_entries.npz"

# Corresponding repo paths (inside the git clone)
_REPO_ARTIFACTS = {
    # local path key              → repo-relative path
    "manifold": "models/FLOW-Grounded_manifold.npz",
    "vocab":    "models/FLOW-Grounded_vocab.npz",
    "pmi":      "models/FLOW-Grounded_pmi.npz",
    "entries":  "models/FLOW-Grounded_entries.npz",
}

# ── Dataset fingerprint ────────────────────────────────────────────────────
# SHA-256 of the corpus config.  If anything changes the fingerprint changes
# → all checkpoints are invalidated → pipeline re-runs from scratch.
_fp_data = json.dumps({
    "n_per_dataset": N_PER_DATASET,
    "total_texts": len(all_texts),
    "sources": sorted(dict(source_counts).items()),
    "v_max": V_MAX,
    "min_count": MIN_COUNT,
    "window_size": WINDOW_SIZE,
    "n_contrast_passes": N_CONTRAST_PASSES,
}, sort_keys=True)
DATASET_FINGERPRINT = hashlib.sha256(_fp_data.encode()).hexdigest()[:16]
print(f"\n  Dataset fingerprint: {DATASET_FINGERPRINT}")


# ── PMI matrix serialisation (compact numpy — ~4 MB for 10K vocab) ────────

def save_pmi_cache(matrix, path):
    """Save a CoOccurrenceMatrix to a compact .npz file."""
    vocab = list(matrix.vocabulary)
    word2idx = {w: i for i, w in enumerate(vocab)}

    # Symmetric PMI
    pmi_w1, pmi_w2, pmi_val = [], [], []
    for (w1, w2), v in matrix._pmi.items():
        pmi_w1.append(word2idx[w1])
        pmi_w2.append(word2idx[w2])
        pmi_val.append(v)

    # Directed PMI
    dpmi_w1, dpmi_w2, dpmi_val = [], [], []
    for (w1, w2), v in matrix._dpmi.items():
        if w1 in word2idx and w2 in word2idx:
            dpmi_w1.append(word2idx[w1])
            dpmi_w2.append(word2idx[w2])
            dpmi_val.append(v)

    # Unigram counts (in vocabulary order)
    unigram = np.array([matrix._unigram_counts.get(w, 0) for w in vocab],
                       dtype=np.uint32)

    np.savez_compressed(
        path,
        vocabulary  = np.array(vocab, dtype=object),
        pmi_w1      = np.array(pmi_w1, dtype=np.uint32),
        pmi_w2      = np.array(pmi_w2, dtype=np.uint32),
        pmi_val     = np.array(pmi_val, dtype=np.float32),
        dpmi_w1     = np.array(dpmi_w1, dtype=np.uint32),
        dpmi_w2     = np.array(dpmi_w2, dtype=np.uint32),
        dpmi_val    = np.array(dpmi_val, dtype=np.float32),
        unigram     = unigram,
    )

def load_pmi_cache(path):
    """Load a CoOccurrenceMatrix from a compact .npz file."""
    data = np.load(path, allow_pickle=True)
    vocab = list(data["vocabulary"])

    pmi_dict = {}
    for i in range(len(data["pmi_val"])):
        w1 = vocab[int(data["pmi_w1"][i])]
        w2 = vocab[int(data["pmi_w2"][i])]
        pmi_dict[(w1, w2)] = float(data["pmi_val"][i])

    dpmi_dict = {}
    for i in range(len(data["dpmi_val"])):
        w1 = vocab[int(data["dpmi_w1"][i])]
        w2 = vocab[int(data["dpmi_w2"][i])]
        dpmi_dict[(w1, w2)] = float(data["dpmi_val"][i])

    unigram_counts = {w: int(c) for w, c in zip(vocab, data["unigram"])}

    return CoOccurrenceMatrix(
        pmi_dict=pmi_dict,
        dpmi_dict=dpmi_dict,
        unigram_counts=unigram_counts,
        vocabulary=vocab,
    )


# ── Checkpoint helpers ─────────────────────────────────────────────────────

def _ckpt_path(step_name: str) -> str:
    """Path to the checkpoint marker for a given step."""
    return f"{CHECKPOINT_DIR}/{step_name}_done.json"

def step_is_done(step_name: str) -> bool:
    """Return True if step was already completed with the current dataset."""
    path = _ckpt_path(step_name)
    if not os.path.isfile(path):
        return False
    try:
        with open(path) as f:
            info = json.load(f)
        return info.get("fingerprint") == DATASET_FINGERPRINT
    except Exception:
        return False

def mark_step_done(step_name: str, **extra):
    """Write a checkpoint marker for a completed step."""
    info = {"fingerprint": DATASET_FINGERPRINT, "step": step_name, **extra}
    with open(_ckpt_path(step_name), "w") as f:
        json.dump(info, f, indent=2, default=str)

def save_manifold_checkpoint(step_name: str):
    """Save manifold + push to GitHub.  Called after each long step."""
    ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)
    _push_to_github(f"checkpoint after {step_name}")

def _push_to_github(msg: str):
    """Stage ALL checkpoint artifacts and push to GitHub."""
    if not PAT:
        print(f"  ⚠ No PAT — checkpoint saved locally only")
        return
    models_dir = f"{REPO_DIR}/models"
    ckpt_repo_dir = f"{REPO_DIR}/checkpoints"
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(ckpt_repo_dir, exist_ok=True)

    # Copy all model artifacts from /kaggle/working/ → repo/models/
    for local_path, repo_rel in [
        (GROUNDED_MANIFOLD_PATH, "models/FLOW-Grounded_manifold.npz"),
        (GROUNDED_VOCAB_PATH,    "models/FLOW-Grounded_vocab.npz"),
        (PMI_CACHE_PATH,         "models/FLOW-Grounded_pmi.npz"),
        (ENTRIES_CACHE_PATH,     "models/FLOW-Grounded_entries.npz"),
    ]:
        if os.path.isfile(local_path):
            shutil.copy2(local_path, f"{REPO_DIR}/{repo_rel}")

    # Copy checkpoint markers (JSON only)
    for f_name in os.listdir(CHECKPOINT_DIR):
        if f_name.endswith(".json"):
            shutil.copy2(f"{CHECKPOINT_DIR}/{f_name}", f"{ckpt_repo_dir}/{f_name}")

    subprocess.run(["git", "config", "user.email", "kaggle@flow.dev"],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "config", "user.name", "FLOW-Kaggle"],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "add", "models/", "checkpoints/"],
                   cwd=REPO_DIR, capture_output=True)
    r = subprocess.run(
        ["git", "commit", "-m", msg],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    if r.returncode == 0:
        subprocess.run(["git", "push", "origin", "main"],
                       cwd=REPO_DIR, capture_output=True, text=True)
        h = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
        print(f"  ✓ Pushed to GitHub ({h}): {msg}")
    elif "nothing to commit" in (r.stdout + r.stderr):
        print(f"  ✓ Nothing new to push")
    else:
        print(f"  ⚠ Git commit issue: {r.stderr.strip()[:120]}")


def restore_from_repo():
    """Copy ALL checkpoint artifacts from the cloned repo back to /kaggle/working/.

    This is the key function that makes resume-after-5-days work:
    Fresh Kaggle session → git clone → this function → all artifacts restored.
    """
    repo_ckpt = f"{REPO_DIR}/checkpoints"
    restored = []

    # 1. Restore checkpoint markers (JSON files)
    if os.path.isdir(repo_ckpt):
        for f_name in os.listdir(repo_ckpt):
            if f_name.endswith(".json"):
                src = f"{repo_ckpt}/{f_name}"
                dst = f"{CHECKPOINT_DIR}/{f_name}"
                if not os.path.isfile(dst):
                    shutil.copy2(src, dst)
                    restored.append(f"  ✓ checkpoint: {f_name}")

    # 2. Restore ALL model artifacts from repo/models/ → /kaggle/working/
    for artifact_name, (local_path, repo_rel) in {
        "manifold": (GROUNDED_MANIFOLD_PATH, "models/FLOW-Grounded_manifold.npz"),
        "vocab":    (GROUNDED_VOCAB_PATH,    "models/FLOW-Grounded_vocab.npz"),
        "pmi":      (PMI_CACHE_PATH,         "models/FLOW-Grounded_pmi.npz"),
        "entries":  (ENTRIES_CACHE_PATH,      "models/FLOW-Grounded_entries.npz"),
    }.items():
        repo_path = f"{REPO_DIR}/{repo_rel}"
        if not os.path.isfile(local_path) and os.path.isfile(repo_path):
            shutil.copy2(repo_path, local_path)
            sz = os.path.getsize(local_path) / 1024 / 1024
            restored.append(f"  ✓ {artifact_name}: {repo_rel} ({sz:.1f} MB)")

    # 3. Load manifold state into the live pipeline
    if os.path.isfile(GROUNDED_MANIFOLD_PATH):
        ManifoldSnapshot.load(GROUNDED_MANIFOLD_PATH, manifold=pipeline.manifold)
        restored.append(f"  ✓ manifold loaded: {pipeline.n_concepts:,} concepts")

    return restored


# ── Restore checkpoints from previous session ──────────────────────────────
print("\nRestoring checkpoints from previous runs...")
restored_items = restore_from_repo()
if restored_items:
    for item in restored_items:
        print(item)
else:
    print("  No prior checkpoint found — starting fresh")

# Show which steps are already done
print()
for step in ["5a_pmi", "5b_placement", "5c_contrast", "5d_radius", "5e_entries", "5f_save"]:
    status = "✓ DONE" if step_is_done(step) else "… pending"
    # Also check if the data file exists
    data_ok = ""
    if step == "5a_pmi":
        data_ok = " [data ✓]" if os.path.isfile(PMI_CACHE_PATH) else " [data MISSING]"
    elif step == "5b_placement":
        data_ok = " [data ✓]" if os.path.isfile(GROUNDED_MANIFOLD_PATH) else " [data MISSING]"
    elif step == "5e_entries":
        data_ok = " [data ✓]" if os.path.isfile(ENTRIES_CACHE_PATH) else " [data MISSING]"
    elif step == "5f_save":
        data_ok = " [data ✓]" if os.path.isfile(GROUNDED_VOCAB_PATH) else " [data MISSING]"
    print(f"  {step:20s} : {status}{data_ok}")

print(f"\n✓ Checkpoint system ready")

In [ ]:
# ── Create VocabularyBuilder ───────────────────────────────
builder = VocabularyBuilder(
    manifold=pipeline.manifold,
    annealing_engine=pipeline._annealing,
    contrast_engine=pipeline._contrast_engine,
    window_size=WINDOW_SIZE,
    min_count=MIN_COUNT,
    v_max=V_MAX,
    tau_same=TAU_SAME,
    tau_diff=TAU_DIFF,
    batch_size=BATCH_SIZE,
    n_contrast_passes=N_CONTRAST_PASSES,
)

print("✓ VocabularyBuilder created")

# Check if corpus feed can be skipped (PMI already built)
if step_is_done("5a_pmi"):
    print("  ✓ PMI already built — corpus feed will be skipped in next cell")
else:
    print("  Corpus feed required — next cell will ingest all texts")

In [ ]:
# ── Feed all corpora ───────────────────────────────────────────
# SKIP LOGIC: Only skip if the PMI cache FILE exists (not just the marker).
# If Kaggle destroyed our session but the PMI was pushed to GitHub, the
# restore_from_repo() function already copied it back.  If the file is
# truly missing, we must re-feed the corpus even if the marker says "done".

_pmi_data_available = os.path.isfile(PMI_CACHE_PATH)

if step_is_done("5a_pmi") and _pmi_data_available:
    print("✓ SKIPPED — PMI cache file available, no need to re-feed corpus")
    print(f"  PMI cache: {PMI_CACHE_PATH} ({os.path.getsize(PMI_CACHE_PATH)/1024/1024:.1f} MB)")
    print("  (Delete the PMI cache + checkpoint to force re-feed)")
else:
    if step_is_done("5a_pmi") and not _pmi_data_available:
        print("⚠ PMI marker says done but data file is MISSING — must re-feed")
        # Reset the marker so 5A rebuilds too
        try: os.remove(_ckpt_path("5a_pmi"))
        except FileNotFoundError: pass

    # Shuffled to interleave domains — prevents the manifold from
    # crystallising on one domain before seeing others.
    import random
    random.seed(42)
    random.shuffle(all_texts)

    t0 = time.time()
    REPORT_EVERY = 2_500
    n_total = len(all_texts)

    for i, (text, source) in enumerate(all_texts):
        builder.feed(text)
        if (i + 1) % REPORT_EVERY == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            print(f"  [{i+1:>6,} / {n_total:,}]  "
                  f"tokens: {builder.n_tokens_fed:>10,}  "
                  f"({rate:.0f} texts/sec, {elapsed:.1f}s)")

    elapsed_feed = time.time() - t0
    print(f"\n✓ All corpora ingested")
    print(f"  Texts processed : {n_total:,}")
    print(f"  Total tokens    : {builder.n_tokens_fed:,}")
    print(f"  Time            : {elapsed_feed:.1f}s")

## 5. Build Vocabulary — 6-step pipeline (checkpoint-resumable)

Each step is **checkpoint-aware**. On completion it pushes all data to
GitHub. If Kaggle disconnects (or you come back 5 days later), re-running
the notebook from the top will:

1. **Clone repo** → all artifacts are already in `models/` and `checkpoints/`
2. **`restore_from_repo()`** → copies them back to `/kaggle/working/`
3. Each step checks: is my marker AND my data file present? → **skip**

### What gets pushed to GitHub (survives across sessions)

| File | Contents | Size (est.) |
|---|---|---|
| `models/FLOW-Grounded_manifold.npz` | Full M(t) state | 5–50 MB |
| `models/FLOW-Grounded_vocab.npz` | Final vocabulary entries | 2–20 MB |
| `models/FLOW-Grounded_pmi.npz` | PMI matrix (compact numpy) | 2–8 MB |
| `models/FLOW-Grounded_entries.npz` | Pre-merge expression entries | 2–20 MB |
| `checkpoints/*.json` | Step-completion markers + fingerprint | < 1 KB each |

### Checkpoint strategy per step

| Step | Duration | Strategy |
|---|---|---|
| **5A** PMI matrix | Seconds | Cache as `.npz` in `models/`; skip rebuild on resume |
| **5B** Word placement | Seconds–minutes | Save manifold snapshot + push |
| **5C** Contrast passes | **Minutes–hours** | Push after **each pass** + every 25K judgments within a pass |
| **5D** Phrase radius | Milliseconds | Save calibrated radius in JSON marker |
| **5E** Build entries | Seconds–minutes | Cache as `.npz` in `models/`; skip rebuild on resume |
| **5F** Merge & save | Seconds | Save final vocab `.npz` + push |

### Dataset fingerprint

SHA-256 of (N_PER_DATASET, source counts, V_MAX, MIN_COUNT, WINDOW_SIZE,
N_CONTRAST_PASSES). If you change any of these, ALL checkpoints are
invalidated and the pipeline re-runs from scratch.

In [ ]:
# ── 5A: Build PMI matrix (checkpoint-aware) ─────────────────
import traceback

print("═" * 70)
print("  STEP 5A — Build PMI matrix")
print("═" * 70)
t0 = time.time()

if step_is_done("5a_pmi") and os.path.isfile(PMI_CACHE_PATH):
    # ── RESUME: Load cached PMI matrix from .npz ────────────
    print(f"  Loading PMI cache: {PMI_CACHE_PATH}")
    print(f"    File size: {os.path.getsize(PMI_CACHE_PATH)/1024/1024:.2f} MB")
    matrix = load_pmi_cache(PMI_CACHE_PATH)
    builder._matrix = matrix
    elapsed_pmi = time.time() - t0
    print(f"  ✓ PMI matrix restored in {elapsed_pmi:.1f}s")
    print(f"  Vocabulary size    : {len(matrix.vocabulary):,}")
    print(f"  PMI pairs          : {len(matrix._pmi):,}")
    print(f"  Directed PMI pairs : {len(matrix._dpmi):,}")
else:
    # ── FRESH BUILD ─────────────────────────────────────────
    try:
        matrix = builder._counter.build()
        builder._matrix = matrix
        elapsed_pmi = time.time() - t0

        print(f"  ✓ PMI matrix built in {elapsed_pmi:.1f}s")
        print(f"  Vocabulary size    : {len(matrix.vocabulary):,}")
        print(f"  PMI pairs          : {len(matrix._pmi):,}")
        print(f"  Directed PMI pairs : {len(matrix._dpmi):,}")
        print(f"  PMI max            : {matrix.pmi_max():.4f}")

        # Save PMI matrix as compact .npz (survives GitHub push)
        save_pmi_cache(matrix, PMI_CACHE_PATH)
        sz = os.path.getsize(PMI_CACHE_PATH) / 1024 / 1024
        print(f"  ✓ PMI cached to {PMI_CACHE_PATH} ({sz:.2f} MB)")

        # Mark step done + push (PMI .npz is in models/, gets pushed)
        mark_step_done("5a_pmi", vocabulary_size=len(matrix.vocabulary),
                       pmi_pairs=len(matrix._pmi))
        _push_to_github(f"5A done: PMI matrix ({len(matrix.vocabulary):,} words)")

    except Exception as e:
        elapsed_pmi = time.time() - t0
        print(f"  ✗ FAILED: {e}")
        traceback.print_exc()
        raise

# Top-10 PMI pairs (always show for verification)
top_pairs = matrix.pairs_above_threshold(1.0, -99)[:10]
print(f"\n  Top 10 PMI pairs:")
for w1, w2, pmi in top_pairs:
    print(f"    PMI={pmi:>7.3f}  {w1} ↔ {w2}")

# Contrast workload preview
same_pairs = [p for p in matrix.pairs_above_threshold(
    builder._scheduler.tau_same, -99) if p[2] > builder._scheduler.tau_same]
diff_pairs = [p for p in matrix.pairs_above_threshold(
    99, builder._scheduler.tau_diff) if p[2] < builder._scheduler.tau_diff]
causal_pairs = matrix.directed_pairs_above_delta(
    builder._scheduler.delta_causal)
print(f"\n  Contrast workload preview:")
print(f"    SAME pairs (PMI > {builder._scheduler.tau_same})  : {len(same_pairs):,}")
print(f"    DIFF pairs (PMI < {builder._scheduler.tau_diff}) : {len(diff_pairs):,}")
print(f"    Causal pairs (δ > {builder._scheduler.delta_causal})   : {len(causal_pairs):,}")
est_total = (len(same_pairs) + len(diff_pairs)) * builder.n_contrast_passes
print(f"    Est. total judgments                : ~{est_total:,}")

print(f"\n{'═' * 70}")
print(f"  5A COMPLETE — {elapsed_pmi:.1f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5B: Place vocabulary words on M(t) (checkpoint-aware) ──
from src.vocabulary.word_placer import WordPlacer, batch_structural_vectors_gpu

print("═" * 70)
print("  STEP 5B — Place vocabulary words on M(t)")
print("═" * 70)

vocab = matrix.vocabulary
freq_ranks = list(range(1, len(vocab) + 1))
n_words = len(vocab)

if step_is_done("5b_placement"):
    # ── RESUME: Words already placed, manifold already restored ──
    print(f"  ✓ SKIPPED — {n_words:,} words already placed on M(t)")
    print(f"  Manifold concepts  : {pipeline.n_concepts:,}")
    elapsed_place = 0.0

    # Verify a spot-check
    for w in vocab[:3]:
        lbl = f"vocab::{w}"
        try:
            pos = pipeline.manifold.position(lbl)
            print(f"    ✓ {lbl:30s}  ‖P‖={np.linalg.norm(pos):.4f}")
        except (KeyError, ValueError):
            print(f"    ✗ {lbl} NOT FOUND — checkpoint may be stale, re-run needed")
            raise RuntimeError(f"Checkpoint stale: {lbl} not on manifold")
else:
    # ── FRESH PLACEMENT ─────────────────────────────────────
    print(f"  Words to place     : {n_words:,}")
    print(f"  Manifold before    : {pipeline.n_concepts:,} concepts")
    print(f"  Temperature        : {pipeline.temperature:.6f}")

    def placement_progress(i, total, label):
        elapsed = time.time() - t_place
        rate = i / elapsed if elapsed > 0 else 0
        pct = 100 * i / total
        print(f"    [{i:>6,}/{total:,}] {pct:5.1f}%  "
              f"{rate:,.0f} words/sec  "
              f"T={pipeline.temperature:.6f}  "
              f"last: {label}")

    t_place = time.time()
    try:
        try:
            import cupy as _cp
            print(f"  → Using GPU-accelerated structural vectors")
            placed = builder._placer.place_batch_gpu(vocab, freq_ranks,
                                                     progress_callback=placement_progress)
        except ImportError:
            print(f"  → Using optimized CPU batch placement")
            placed = builder._placer.place_batch(vocab, freq_ranks,
                                                 progress_callback=placement_progress)

        builder._n_words_placed = len(vocab)
        elapsed_place = time.time() - t_place

        print(f"\n  ✓ Word placement complete")
        print(f"  Words placed       : {len(placed):,}")
        print(f"  Manifold after     : {pipeline.n_concepts:,} concepts")
        print(f"  Time               : {elapsed_place:.1f}s ({elapsed_place/60:.1f} min)")
        print(f"  Rate               : {len(placed)/elapsed_place:,.0f} words/sec")

        # Spot-check
        for w in vocab[:3]:
            lbl = f"vocab::{w}"
            try:
                pos = pipeline.manifold.position(lbl)
                d = pipeline.manifold.density(pos)
                print(f"    ✓ {lbl:30s}  ρ={d:.4f}  ‖P‖={np.linalg.norm(pos):.4f}")
            except (KeyError, ValueError) as e:
                print(f"    ✗ {lbl} — NOT FOUND: {e}")

        # ── CHECKPOINT: Save manifold + mark done ────────────
        print(f"\n  Saving checkpoint...")
        save_manifold_checkpoint("5b_placement")
        mark_step_done("5b_placement", words_placed=len(placed),
                       manifold_concepts=pipeline.n_concepts)
        print(f"  ✓ Checkpoint saved & pushed")

    except Exception as e:
        elapsed_place = time.time() - t_place
        print(f"\n  ✗ FAILED after {elapsed_place:.1f}s: {e}")
        # Save partial progress even on failure
        print(f"  Saving partial manifold for recovery...")
        try:
            ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)
            _push_to_github(f"5B PARTIAL: {pipeline.n_concepts:,} concepts (interrupted)")
        except Exception:
            pass
        traceback.print_exc()
        raise

print(f"\n{'═' * 70}")
print(f"  5B COMPLETE — {elapsed_place:.1f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5C: Contrast refinement — C4 SAME/DIFFERENT (checkpoint per-pass) ──
#
# This is the longest step.  We checkpoint after EACH contrast pass so that
# if Kaggle disconnects mid-way, we resume from the last completed pass
# instead of re-running the entire thing.
#
# Additionally, within each pass we checkpoint every INTRA_PASS_CHECKPOINT
# judgments to handle very large pair sets within a single pass.

print("═" * 70)
print("  STEP 5C — Contrast refinement (C4) — with per-pass checkpoints")
print("═" * 70)

INTRA_PASS_CHECKPOINT = 25_000  # save manifold every N judgments within a pass

if step_is_done("5c_contrast"):
    # ── FULLY DONE: Skip entire step ──────────────────────
    ckpt_info = json.load(open(_ckpt_path("5c_contrast")))
    n_judgments_total = ckpt_info.get("total_judgments", 0)
    elapsed_contrast = 0.0
    builder._n_judgments = n_judgments_total
    print(f"  ✓ SKIPPED — All {N_CONTRAST_PASSES} contrast passes already complete")
    print(f"  Total judgments    : {n_judgments_total:,}")
else:
    # ── Determine which passes are already done ──────────
    completed_passes = 0
    cumulative_judgments = 0
    for p in range(1, N_CONTRAST_PASSES + 1):
        if step_is_done(f"5c_pass_{p}"):
            completed_passes = p
            ckpt = json.load(open(_ckpt_path(f"5c_pass_{p}")))
            cumulative_judgments = ckpt.get("cumulative_judgments", 0)

    if completed_passes > 0:
        print(f"  Resuming from pass {completed_passes + 1} "
              f"(passes 1–{completed_passes} done, {cumulative_judgments:,} judgments)")
    else:
        print(f"  Starting fresh — {N_CONTRAST_PASSES} passes to run")

    print(f"  Batch size         : {builder._scheduler.batch_size}")
    print(f"  τ_same             : {builder._scheduler.tau_same}")
    print(f"  τ_diff             : {builder._scheduler.tau_diff}")
    print(f"  Manifold labels    : {len(pipeline.manifold.labels):,}")
    print(f"  Intra-pass ckpt    : every {INTRA_PASS_CHECKPOINT:,} judgments")

    # ── Per-pass loop with checkpointing ────────────────────
    t_contrast = time.time()
    n_judgments_total = cumulative_judgments

    for pass_num in range(completed_passes + 1, N_CONTRAST_PASSES + 1):
        if step_is_done(f"5c_pass_{pass_num}"):
            print(f"\n  Pass {pass_num}/{N_CONTRAST_PASSES} — already done, skipping")
            continue

        print(f"\n  ── Pass {pass_num}/{N_CONTRAST_PASSES} ──────────────────────")
        t_pass = time.time()

        # Check for intra-pass checkpoint (partial pass recovery)
        intra_ckpt_name = f"5c_pass_{pass_num}_partial"
        intra_applied = 0
        if step_is_done(intra_ckpt_name):
            ckpt = json.load(open(_ckpt_path(intra_ckpt_name)))
            intra_applied = ckpt.get("pairs_applied_this_pass", 0)
            print(f"  Resuming intra-pass from pair #{intra_applied:,}")

        # We manually drive the contrast scheduler for checkpoint control.
        # Instead of calling run_passes(), we call run() once per pass
        # with a custom progress callback that checkpoints periodically.
        manifold = pipeline.manifold
        pmi_max  = matrix.pmi_max()
        manifold_labels = set(manifold.labels)

        # ── Phase 1: Symmetric judgments ─────────────────────
        pairs = matrix.pairs_above_threshold(
            builder._scheduler.tau_same, builder._scheduler.tau_diff
        )
        n_total_pairs = len(pairs)
        n_applied_this_pass = 0
        n_skipped_resume = 0
        batch = []
        _last_ckpt_n = [0]
        _last_ckpt_time = [time.time()]

        for idx, (w1, w2, pmi_val) in enumerate(pairs):
            # Skip already-applied pairs from intra-pass checkpoint
            if idx < intra_applied:
                n_skipped_resume += 1
                continue

            la = f"vocab::{w1}"
            lb = f"vocab::{w2}"
            if la not in manifold_labels or lb not in manifold_labels:
                continue

            j = JudgmentType.SAME if pmi_val > builder._scheduler.tau_same \
                else JudgmentType.DIFFERENT
            strength = min(1.0, abs(pmi_val) / pmi_max)
            batch.append(ContrastPair(la, lb, j, strength))

            if len(batch) >= builder._scheduler.batch_size:
                n_applied_this_pass += builder._scheduler._flush_batch(batch)
                batch = []

                # Progress logging (every 10s)
                now = time.time()
                if now - _last_ckpt_time[0] >= 10.0:
                    elapsed = now - t_pass
                    rate = (n_applied_this_pass + n_skipped_resume) / elapsed if elapsed > 0 else 0
                    pct = 100 * (idx + 1) / n_total_pairs
                    print(f"    [{idx+1:>8,} / {n_total_pairs:,}] {pct:5.1f}%  "
                          f"{rate:,.0f} pairs/sec  "
                          f"applied={n_applied_this_pass:,}")
                    _last_ckpt_time[0] = now

                # ── Intra-pass checkpoint ────────────────────
                if (n_applied_this_pass - _last_ckpt_n[0]) >= INTRA_PASS_CHECKPOINT:
                    print(f"    ── Intra-pass checkpoint at {n_applied_this_pass:,} judgments...")
                    ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)
                    mark_step_done(intra_ckpt_name,
                                   pairs_applied_this_pass=idx + 1,
                                   judgments_this_pass=n_applied_this_pass)
                    _push_to_github(
                        f"5C pass {pass_num} partial: {n_applied_this_pass:,} judgments"
                    )
                    _last_ckpt_n[0] = n_applied_this_pass

        # Flush remaining batch
        if batch:
            n_applied_this_pass += builder._scheduler._flush_batch(batch)

        # Rebuild tree after symmetric phase
        manifold.force_rebuild_tree()

        # ── Phase 2: Directed causal fiber bias ─────────────
        directed = matrix.directed_pairs_above_delta(builder._scheduler.delta_causal)
        n_causal = builder._scheduler._apply_causal_bias(
            directed, manifold, manifold_labels
        )
        n_applied_this_pass += n_causal

        # Rebuild tree between passes
        manifold.force_rebuild_tree()

        elapsed_pass = time.time() - t_pass
        n_judgments_total += n_applied_this_pass
        print(f"    ── Pass {pass_num} done: {n_applied_this_pass:,} judgments "
              f"({elapsed_pass:.0f}s, {n_causal:,} causal)")

        # ── PER-PASS CHECKPOINT ─────────────────────────────
        print(f"    Saving pass checkpoint...")
        ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)
        mark_step_done(f"5c_pass_{pass_num}",
                       judgments_this_pass=n_applied_this_pass,
                       cumulative_judgments=n_judgments_total,
                       elapsed_seconds=elapsed_pass)
        # Clean up intra-pass checkpoint (no longer needed)
        intra_path = _ckpt_path(intra_ckpt_name)
        if os.path.isfile(intra_path):
            os.remove(intra_path)
        _push_to_github(
            f"5C pass {pass_num}/{N_CONTRAST_PASSES}: "
            f"{n_applied_this_pass:,} judgments (cumul {n_judgments_total:,})"
        )

    # ── Mark full contrast step done ─────────────────────
    elapsed_contrast = time.time() - t_contrast
    builder._n_judgments = n_judgments_total
    mark_step_done("5c_contrast", total_judgments=n_judgments_total,
                   elapsed_seconds=elapsed_contrast)
    save_manifold_checkpoint("5c_contrast_final")

print(f"\n  ✓ Contrast refinement complete")
print(f"  Total judgments    : {n_judgments_total:,}")
if elapsed_contrast > 0:
    print(f"  Time               : {elapsed_contrast:.1f}s ({elapsed_contrast/60:.1f} min)")

# Spot-check geometry change
if len(vocab) >= 2:
    w1, w2 = vocab[0], vocab[1]
    d = float(np.linalg.norm(
        pipeline.manifold.position(f"vocab::{w1}") -
        pipeline.manifold.position(f"vocab::{w2}")
    ))
    print(f"  Sample dist({w1}, {w2}) = {d:.4f}")

print(f"\n{'═' * 70}")
print(f"  5C COMPLETE")
print(f"{'═' * 70}")

In [ ]:
# ── 5D: Calibrate phrase radius (checkpoint-aware) ──────────
from src.vocabulary.template_builder import TemplateBuilder

print("═" * 70)
print("  STEP 5D — Calibrate phrase radius")
print("═" * 70)

t_calib = time.time()

if step_is_done("5d_radius"):
    ckpt = json.load(open(_ckpt_path("5d_radius")))
    radius = ckpt.get("radius", 0.5)
    tb = TemplateBuilder(pipeline.manifold)
    tb._phrase_radius = radius
    elapsed_calib = time.time() - t_calib
    print(f"  ✓ SKIPPED — radius already calibrated: {radius:.4f}")
else:
    try:
        tb = TemplateBuilder(pipeline.manifold)
        radius = tb.calibrate_phrase_radius()
        elapsed_calib = time.time() - t_calib

        print(f"  ✓ Phrase radius calibrated")
        print(f"  Radius             : {radius:.4f}")
        print(f"  Time               : {elapsed_calib:.2f}s")

        vocab_labels = [l for l in pipeline.manifold._points if l.startswith("vocab::")]
        print(f"  Vocab on manifold  : {len(vocab_labels):,}")

        mark_step_done("5d_radius", radius=radius)

    except Exception as e:
        elapsed_calib = time.time() - t_calib
        print(f"\n  ✗ FAILED after {elapsed_calib:.2f}s: {e}")
        traceback.print_exc()
        raise

print(f"\n{'═' * 70}")
print(f"  5D COMPLETE — {elapsed_calib:.2f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5E: Build ExpressionEntry objects (Level 1 + 2 + 3) — checkpoint-aware ─

print("═" * 70)
print("  STEP 5E — Build ExpressionEntry objects")
print("═" * 70)

t_build = time.time()

if step_is_done("5e_entries") and os.path.isfile(ENTRIES_CACHE_PATH):
    # ── RESUME: Load cached entries from .npz ───────────────
    print(f"  Loading entries cache: {ENTRIES_CACHE_PATH}")
    print(f"    File size: {os.path.getsize(ENTRIES_CACHE_PATH)/1024/1024:.2f} MB")
    new_entries = VocabularyStore.load(ENTRIES_CACHE_PATH)
    elapsed_build = time.time() - t_build
    print(f"  ✓ Restored {len(new_entries):,} entries in {elapsed_build:.1f}s")
else:
    if step_is_done("5e_entries") and not os.path.isfile(ENTRIES_CACHE_PATH):
        print("  ⚠ Entries marker says done but data file is MISSING — rebuilding")

    try:
        # Level 1 — single words
        print("  Building Level 1 (single words)...")
        t_l1 = time.time()
        l1_entries = tb._build_level1()
        elapsed_l1 = time.time() - t_l1
        print(f"    ✓ Level 1: {len(l1_entries):,} entries in {elapsed_l1:.1f}s")
        if l1_entries:
            s = l1_entries[0]
            print(f"    Sample: '{s.text}' register={s.register} "
                  f"causal={s.causal_strength:.3f} hedging={s.hedging}")

        # Level 2 — short phrases
        print("  Building Level 2 (short phrases)...")
        t_l2 = time.time()
        l2_entries = tb._build_level2(matrix)
        elapsed_l2 = time.time() - t_l2
        print(f"    ✓ Level 2: {len(l2_entries):,} entries in {elapsed_l2:.1f}s")
        if l2_entries:
            s = l2_entries[0]
            print(f"    Sample: '{s.text}' register={s.register}")

        # Level 3 — sentence frames
        print("  Building Level 3 (sentence frames)...")
        t_l3 = time.time()
        l3_entries = tb._build_level3()
        elapsed_l3 = time.time() - t_l3
        print(f"    ✓ Level 3: {len(l3_entries):,} entries in {elapsed_l3:.1f}s")
        if l3_entries:
            s = l3_entries[0]
            print(f"    Sample: '{s.text}'")

        # Deduplicate
        new_entries = []
        seen_texts = set()
        for e in l1_entries + l2_entries + l3_entries:
            if e.text not in seen_texts:
                seen_texts.add(e.text)
                new_entries.append(e)

        elapsed_build = time.time() - t_build

        print(f"\n  ✓ All entries built and deduplicated")
        print(f"  Level 1            : {len(l1_entries):,}  ({elapsed_l1:.1f}s)")
        print(f"  Level 2            : {len(l2_entries):,}  ({elapsed_l2:.1f}s)")
        print(f"  Level 3            : {len(l3_entries):,}  ({elapsed_l3:.1f}s)")
        print(f"  New entries (dedup): {len(new_entries):,}")
        print(f"  Total time         : {elapsed_build:.1f}s ({elapsed_build/60:.1f} min)")

        # Register distribution
        reg_dist = Counter(e.register for e in new_entries)
        hedging_count = sum(1 for e in new_entries if e.hedging)
        print(f"\n  Register distribution:")
        for reg, cnt in reg_dist.most_common():
            print(f"    {reg:10s} : {cnt:>6,}")
        print(f"  Hedging entries    : {hedging_count:,}")

        # Cache entries as .npz (pushed to GitHub via _push_to_github)
        VocabularyStore.save(new_entries, ENTRIES_CACHE_PATH)
        sz = os.path.getsize(ENTRIES_CACHE_PATH) / 1024 / 1024
        mark_step_done("5e_entries", n_entries=len(new_entries))
        _push_to_github(f"5E done: {len(new_entries):,} entries ({sz:.1f} MB)")
        print(f"\n  ✓ Entries cached & pushed ({sz:.1f} MB)")

    except Exception as e:
        elapsed_build = time.time() - t_build
        print(f"\n  ✗ FAILED after {elapsed_build:.1f}s: {e}")
        traceback.print_exc()
        raise

print(f"\n{'═' * 70}")
print(f"  5E COMPLETE — {elapsed_build:.1f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5F: Merge with base vocabulary & save (checkpoint-aware) ───

print("═" * 70)
print("  STEP 5F — Merge & save vocabulary")
print("═" * 70)

t_save = time.time()

if step_is_done("5f_save") and os.path.isfile(GROUNDED_VOCAB_PATH):
    # ── SKIP: Already merged & saved ──────────────────────
    ckpt = json.load(open(_ckpt_path("5f_save")))
    n_entries = ckpt.get("n_entries", 0)
    elapsed_save = 0.0
    # Reconstruct base_kept count for summary cell
    new_texts = {e.text for e in new_entries}
    base_kept = [e for e in base_vocab_entries if e.text not in new_texts]
    print(f"  ✓ SKIPPED — vocabulary already saved ({n_entries:,} entries)")
    print(f"  File: {GROUNDED_VOCAB_PATH}")
else:
    try:
        # Merge: new entries take priority over base entries
        new_texts = {e.text for e in new_entries}
        base_kept = [e for e in base_vocab_entries if e.text not in new_texts]
        merged_entries = new_entries + base_kept

        print(f"  New PMI entries    : {len(new_entries):,}")
        print(f"  Base entries kept  : {len(base_kept):,} (of {n_base_vocab:,})")
        print(f"  Base entries replaced: {n_base_vocab - len(base_kept):,}")
        print(f"  Merged total       : {len(merged_entries):,}")

        if not merged_entries:
            raise RuntimeError("No entries to save — check steps 5A–5E.")

        n_entries = VocabularyStore.save(merged_entries, GROUNDED_VOCAB_PATH)
        elapsed_save = time.time() - t_save

        print(f"\n  ✓ Vocabulary saved")
        print(f"  Entries written    : {n_entries:,}")
        print(f"  File               : {GROUNDED_VOCAB_PATH}")
        print(f"  File size          : {os.path.getsize(GROUNDED_VOCAB_PATH) / 1024 / 1024:.2f} MB")
        print(f"  Time               : {elapsed_save:.2f}s")

        mark_step_done("5f_save", n_entries=n_entries)
        save_manifold_checkpoint("5f_save_final")

    except Exception as e:
        elapsed_save = time.time() - t_save
        print(f"\n  ✗ FAILED after {elapsed_save:.2f}s: {e}")
        traceback.print_exc()
        raise

# ── Compute timing from checkpoint metadata ──────────────────
# Gather timing from checkpoints for the summary table
_timing = {}
for step, key in [("5a_pmi", "elapsed_pmi"), ("5b_placement", "elapsed_place"),
                  ("5c_contrast", "elapsed_contrast"), ("5d_radius", "elapsed_calib"),
                  ("5e_entries", "elapsed_build"), ("5f_save", "elapsed_save")]:
    try:
        _t = json.load(open(_ckpt_path(step))).get("elapsed_seconds", 0)
        _timing[key] = _t
    except Exception:
        pass
elapsed_pmi      = _timing.get("elapsed_pmi", locals().get("elapsed_pmi", 0))
elapsed_place    = _timing.get("elapsed_place", locals().get("elapsed_place", 0))
elapsed_contrast = _timing.get("elapsed_contrast", locals().get("elapsed_contrast", 0))
elapsed_calib    = _timing.get("elapsed_calib", locals().get("elapsed_calib", 0))
elapsed_build    = _timing.get("elapsed_build", locals().get("elapsed_build", 0))
elapsed_save     = _timing.get("elapsed_save", locals().get("elapsed_save", 0))
total_build_time = elapsed_pmi + elapsed_place + elapsed_contrast + elapsed_calib + elapsed_build + elapsed_save

print(f"\n{'═' * 70}")
print(f"  5F COMPLETE — vocabulary saved")
print(f"{'═' * 70}")
print(f"\n  ┌─────────────────────────────────────────────────┐")
print(f"  │  BUILD TIMING SUMMARY                           │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  5A PMI matrix      : {elapsed_pmi:>8.1f}s  ({elapsed_pmi/60:>5.1f} min)  │")
print(f"  │  5B Word placement  : {elapsed_place:>8.1f}s  ({elapsed_place/60:>5.1f} min)  │")
print(f"  │  5C Contrast passes : {elapsed_contrast:>8.1f}s  ({elapsed_contrast/60:>5.1f} min)  │")
print(f"  │  5D Phrase radius   : {elapsed_calib:>8.1f}s  ({elapsed_calib/60:>5.1f} min)  │")
print(f"  │  5E Build entries   : {elapsed_build:>8.1f}s  ({elapsed_build/60:>5.1f} min)  │")
print(f"  │  5F Save .npz       : {elapsed_save:>8.1f}s  ({elapsed_save/60:>5.1f} min)  │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  TOTAL              : {total_build_time:>8.1f}s  ({total_build_time/60:>5.1f} min)  │")
print(f"  └─────────────────────────────────────────────────┘")

In [ ]:
# ── Save manifold state (final — always runs to ensure latest) ──

n_saved = ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)

print(f"✓ Manifold state saved")
print(f"  Concepts on M(t)   : {n_saved:,}")
print(f"  File               : {GROUNDED_MANIFOLD_PATH}")
print(f"  File size          : {os.path.getsize(GROUNDED_MANIFOLD_PATH) / 1024 / 1024:.2f} MB")
print(f"  Temperature        : {pipeline.temperature:.6f}")

# Also ensure n_entries is defined for the summary cell
if 'n_entries' not in dir():
    if os.path.isfile(GROUNDED_VOCAB_PATH):
        n_entries = VocabularyStore.count(GROUNDED_VOCAB_PATH) if hasattr(VocabularyStore, 'count') else len(VocabularyStore.load(GROUNDED_VOCAB_PATH))

## 6. Verify Artifacts

In [ ]:
# ── Verify manifold round-trip ─────────────────────────────
info = ManifoldSnapshot.info(GROUNDED_MANIFOLD_PATH)
print("Manifold snapshot info:")
for k, v in info.items():
    print(f"  {k:20s} : {v}")

# Spot-check: reload and compare positions
reloaded = ManifoldSnapshot.load(GROUNDED_MANIFOLD_PATH)
labels = list(pipeline.manifold._points.keys())
max_err = 0.0
for label in labels[:100]:
    orig = pipeline.manifold.position(label)
    rest = reloaded.position(label)
    err = np.max(np.abs(orig - rest))
    max_err = max(max_err, err)

print(f"\n✓ Manifold round-trip verified")
print(f"  Max position error : {max_err:.2e}")
assert max_err < 1e-10, f"Round-trip error too large: {max_err}"

# Verify vocabulary
loaded_entries = VocabularyStore.load(GROUNDED_VOCAB_PATH)
print(f"\nVocabulary verification:")
print(f"  Entries loaded     : {len(loaded_entries):,}")
print(f"\n  Sample entries (first 20):")
for entry in loaded_entries[:20]:
    print(f"    [{entry.register:>7s}] [{entry.rhythm:>6s}] {entry.text[:60]}")

## 7. Evaluate — Query the Grounded Manifold

In [ ]:
# ── Load grounded vocabulary into the renderer ───────────────
# Use extend + _faiss_dirty to load in-memory entries
pipeline._renderer.matcher.vocabulary.clear()
pipeline._renderer.matcher.vocabulary.extend(loaded_entries)
pipeline._renderer.matcher._faiss_dirty = True
print(f"✓ Loaded {len(loaded_entries):,} entries into C7 renderer")

# ── Sample queries ───────────────────────────────────────
all_labels = list(pipeline.manifold._points.keys())
vocab_labels = [l for l in all_labels if l.startswith("vocab::")]
print(f"Vocabulary concepts on manifold: {len(vocab_labels):,}")
print(f"Total concepts on manifold     : {len(all_labels):,}")

sample_words = vocab_labels[:10] if len(vocab_labels) >= 10 else vocab_labels
print(f"\nSample queries:")
print("=" * 70)

for label in sample_words:
    vec = pipeline.manifold.position(label)
    result = pipeline.query(vec, label=label)
    word = label.replace("vocab::", "")
    print(f"\n  Query: {word}")
    print(f"  Steps: {result.n_steps}  |  Reason: {result.termination_reason}")
    print(f"  Confidence: {result.confidence:.3f}  |  Wave: {result.wave_confidence:.3f}")
    print(f"  Output: {result.text[:120]}")
    print("-" * 70)

In [ ]:
# ── Evaluation suite ─────────────────────────────────────
from src.phase5 import PipelineEvaluator

evaluator = PipelineEvaluator(pipeline)

n_eval = min(50, len(vocab_labels))
eval_labels = vocab_labels[:n_eval]
eval_vecs = [pipeline.manifold.position(l) for l in eval_labels]

print(f"Running evaluation suite with {n_eval} queries...")
t0 = time.time()
suite = evaluator.run_suite(eval_vecs, eval_labels)
elapsed_eval = time.time() - t0

print(f"\n✓ Evaluation complete ({elapsed_eval:.1f}s)")
print(f"  Mean coherence       : {suite.mean_coherence:.4f}")
print(f"  Novelty is decaying  : {suite.novelty_is_decaying}")
print(f"  Termination reasons  :")
for reason, count in suite.termination_distribution.items():
    print(f"    {reason:30s} : {count}")

## 8. Summary

In [ ]:
# ── Final summary ─────────────────────────────────────────
# Use locals with fallbacks for variables that may not exist on checkpoint resume
_n_entries = n_entries if 'n_entries' in dir() else '?'
_n_tokens = builder.n_tokens_fed if hasattr(builder, 'n_tokens_fed') and builder.n_tokens_fed > 0 else '(from checkpoint)'
_n_judgments = builder.n_judgments_applied if hasattr(builder, 'n_judgments_applied') else n_judgments_total if 'n_judgments_total' in dir() else '?'
_elapsed_feed = elapsed_feed if 'elapsed_feed' in dir() else 0

print("=" * 70)
print("  FLOW — Vocabulary Growth (Corpus-Grounded) — Final Report")
print("=" * 70)
print(f"")
print(f"  Base model: {base_name}")
print(f"  Dataset fingerprint: {DATASET_FINGERPRINT}")
print(f"")
print(f"  Corpora")
print(f"    Wikipedia          : {source_counts.get('wikipedia', 0):,}")
print(f"    OpenAssistant      : {source_counts.get('openassistant', 0):,}")
print(f"    Alpaca (cleaned)   : {source_counts.get('alpaca', 0):,}")
print(f"    Glaive Functions   : {source_counts.get('glaive_functions', 0):,}")
print(f"    SlimOrca           : {source_counts.get('slimorca', 0):,}")
print(f"    Total texts        : {len(all_texts):,}")
print(f"    Total tokens       : {_n_tokens}")
print(f"")
print(f"  Vocabulary")
print(f"    New PMI entries     : {len(new_entries):,}")
print(f"    Base entries kept   : {len(base_kept):,}")
print(f"    Merged total        : {_n_entries}")
print(f"    Contrast judgments  : {_n_judgments}")
print(f"    vocab.npz size     : {os.path.getsize(GROUNDED_VOCAB_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Manifold")
print(f"    Total concepts     : {pipeline.n_concepts:,}")
print(f"    Dimension          : {pipeline.manifold.DIM}")
print(f"    Temperature        : {pipeline.temperature:.6f}")
print(f"    manifold.npz size  : {os.path.getsize(GROUNDED_MANIFOLD_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Evaluation")
print(f"    Mean coherence     : {suite.mean_coherence:.4f}")
print(f"    Novelty decaying   : {suite.novelty_is_decaying}")
print(f"")
print(f"  Build time           : {total_build_time:.0f}s ({total_build_time/60:.1f} min)")
print(f"  Corpus feed time     : {_elapsed_feed}")
print(f"")
print(f"  Checkpoint status:")
for step in ["5a_pmi", "5b_placement", "5c_contrast", "5d_radius", "5e_entries", "5f_save"]:
    status = "✓" if step_is_done(step) else "✗"
    print(f"    {status} {step}")
print("=" * 70)

## 9. Save & Push to GitHub

In [ ]:
# ── Final push to GitHub (all steps complete) ─────────────────
import shutil

if PAT:
    models_dir = f"{REPO_DIR}/models"
    ckpt_repo_dir = f"{REPO_DIR}/checkpoints"
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(ckpt_repo_dir, exist_ok=True)

    # Copy ALL artifacts into repo
    for local_path, repo_rel in [
        (GROUNDED_MANIFOLD_PATH, "models/FLOW-Grounded_manifold.npz"),
        (GROUNDED_VOCAB_PATH,    "models/FLOW-Grounded_vocab.npz"),
        (PMI_CACHE_PATH,         "models/FLOW-Grounded_pmi.npz"),
        (ENTRIES_CACHE_PATH,     "models/FLOW-Grounded_entries.npz"),
    ]:
        if os.path.isfile(local_path):
            shutil.copy2(local_path, f"{REPO_DIR}/{repo_rel}")

    # Copy checkpoint markers (JSON only)
    for f_name in os.listdir(CHECKPOINT_DIR):
        if f_name.endswith(".json"):
            shutil.copy2(f"{CHECKPOINT_DIR}/{f_name}", f"{ckpt_repo_dir}/{f_name}")
    print(f"✓ Artifacts copied to {models_dir}/")

    # Configure git
    subprocess.run(["git", "config", "user.email", "kaggle@flow.dev"],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "config", "user.name", "FLOW-Kaggle"],
                   cwd=REPO_DIR, capture_output=True)

    # Stage, commit, push
    cmds = [
        ["git", "add", "models/", "checkpoints/"],
        ["git", "commit", "-m",
         f"Grounded vocab FINAL: {n_entries:,} entries, "
         f"{pipeline.n_concepts:,} concepts, "
         f"{builder.n_tokens_fed:,} tokens from 5 corpora"],
        ["git", "push", "origin", "main"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if r.returncode != 0 and "nothing to commit" not in (r.stdout + r.stderr):
            print(f"  ✘ {' '.join(cmd[:3])}: {r.stderr.strip()[:120]}")

    final_hash = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    print(f"✓ Final push to GitHub (commit {final_hash})")

    # Show what's in model storage
    for f_name in sorted(os.listdir(models_dir)):
        if f_name.startswith("FLOW-Grounded"):
            sz = os.path.getsize(f"{models_dir}/{f_name}") / 1024 / 1024
            print(f"  {f_name:45s} {sz:>6.1f} MB")
else:
    print("⚠ No PAT — artifacts saved to /kaggle/working/ only")
    print(f"  {GROUNDED_MANIFOLD_PATH}")
    print(f"  {GROUNDED_VOCAB_PATH}")

print(f"\nTo reload locally:")
print(f"  pipeline = GEOPipeline.load(")
print(f"      'FLOW-Grounded_manifold.npz',")
print(f"      vocabulary_path='FLOW-Grounded_vocab.npz')")